In [ ]:
# ============================================
# CYCLEGAN - 100 EPOCHS (SHAPE-FIXED)
# Run this second
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.utils as vutils
import numpy as np
from tqdm import tqdm
import os
import random
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import torchvision.transforms as transforms

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configuration
class Config:
    img_size = 64
    batch_size = 4
    dataset_path = '/content/drive/MyDrive/CSE720/EyeGAN'
    class_names = {
        0: 'Diabetic Retinopathy',
        1: 'Glaucoma',
        2: 'Healthy',
        3: 'Macular Scar',
        4: 'Myopia'
    }

cfg = Config()

# ============================================
# CYCLEGAN ARCHITECTURE
# ============================================

class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, in_channels, 3),
            nn.InstanceNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, in_channels, 3),
            nn.InstanceNorm2d(in_channels)
        )

    def forward(self, x):
        return x + self.block(x)


class CycleGANGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_residual_blocks=6):
        super(CycleGANGenerator, self).__init__()

        # Initial convolution block
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, 64, 7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True)
        ]

        # Downsampling
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, 3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2

        # Residual blocks
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_features)]

        # Upsampling
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2

        # Output layer
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, out_channels, 7),
            nn.Tanh()
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


class CycleGANDiscriminator(nn.Module):
    def __init__(self, in_channels=3):
        super(CycleGANDiscriminator, self).__init__()

        def block(in_filters, out_filters, normalize=True):
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        # Calculate output size for 64x64 input
        # After 4 downsampling layers: 64 -> 32 -> 16 -> 8 -> 4
        # Then final conv with padding=1 gives 4x4 output
        self.model = nn.Sequential(
            *block(in_channels, 64, normalize=False),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.Conv2d(512, 1, 4, padding=1)  # Output: 4x4
        )

    def forward(self, x):
        return self.model(x)


class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):
        to_return = []
        for element in data:
            element = element.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(element)
                to_return.append(element)
            else:
                if random.random() > 0.5:
                    i = random.randint(0, self.max_size - 1)
                    to_return.append(self.data[i].clone())
                    self.data[i] = element
                else:
                    to_return.append(element)
        return torch.cat(to_return)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ============================================
# DATASET CLASSES
# ============================================

class UnpairedDataset(Dataset):
    def __init__(self, domain_path, img_size=64):
        self.images = []
        for f in os.listdir(domain_path):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                self.images.append(os.path.join(domain_path, f))

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        print(f"Loaded {len(self.images)} images from {domain_path}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        return self.transform(img)


class MedicalTestDataset(Dataset):
    def __init__(self, root_dir, image_size=64):
        self.root_dir = root_dir
        self.image_size = image_size
        self.domains = sorted([d.strip() for d in os.listdir(root_dir)
                             if os.path.isdir(os.path.join(root_dir, d))])
        self.image_paths = []
        self.labels = []
        self.domain_to_label = {domain: idx for idx, domain in enumerate(self.domains)}

        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            if not os.path.exists(domain_path):
                continue
            domain_images = [os.path.join(domain_path, img) for img in os.listdir(domain_path)
                           if img.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
            total_images = len(domain_images)
            test_split = int(total_images * 0.9)
            selected_images = domain_images[test_split:]
            label = self.domain_to_label[domain]
            self.image_paths.extend(selected_images)
            self.labels.extend([label] * len(selected_images))

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        print(f"Test dataset: {len(self.image_paths)} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = self.labels[idx]
        return image, label, img_path


# ============================================
# TRAINING FUNCTION
# ============================================

def train_cyclegan(source_name, target_name, epochs=100):
    """Train CycleGAN for unpaired image translation"""

    save_dir = f'/content/drive/MyDrive/CSE720/cyclegan_results'
    os.makedirs(save_dir, exist_ok=True)

    # Create datasets
    source_path = os.path.join(cfg.dataset_path, source_name)
    target_path = os.path.join(cfg.dataset_path, target_name)

    source_dataset = UnpairedDataset(source_path, cfg.img_size)
    target_dataset = UnpairedDataset(target_path, cfg.img_size)

    source_loader = DataLoader(source_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=0, drop_last=True)
    target_loader = DataLoader(target_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=0, drop_last=True)

    print(f"Source ({source_name}) training samples: {len(source_dataset)}")
    print(f"Target ({target_name}) training samples: {len(target_dataset)}")

    # Initialize models
    G_AtoB = CycleGANGenerator().to(device)
    G_BtoA = CycleGANGenerator().to(device)
    D_A = CycleGANDiscriminator().to(device)
    D_B = CycleGANDiscriminator().to(device)

    G_AtoB.apply(weights_init)
    G_BtoA.apply(weights_init)
    D_A.apply(weights_init)
    D_B.apply(weights_init)

    print(f"\nGenerator parameters: {sum(p.numel() for p in G_AtoB.parameters()):,}")
    print(f"Discriminator parameters: {sum(p.numel() for p in D_A.parameters()):,}")

    # Loss functions
    criterion_GAN = nn.MSELoss()
    criterion_cycle = nn.L1Loss()
    criterion_identity = nn.L1Loss()

    # Optimizers
    optimizer_G = optim.Adam(
        list(G_AtoB.parameters()) + list(G_BtoA.parameters()),
        lr=0.0002, betas=(0.5, 0.999)
    )
    optimizer_D_A = optim.Adam(D_A.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optimizer_D_B = optim.Adam(D_B.parameters(), lr=0.0002, betas=(0.5, 0.999))

    # Replay buffers
    fake_A_buffer = ReplayBuffer()
    fake_B_buffer = ReplayBuffer()

    lambda_cycle = 10
    lambda_identity = 5

    # Get discriminator output size
    with torch.no_grad():
        dummy = torch.randn(1, 3, 64, 64).to(device)
        dummy_out = D_A(dummy)
        patch_size = dummy_out.shape[2]  # Should be 4 for 64x64 input
        print(f"Discriminator output patch size: {patch_size}x{patch_size}")

    print("\n" + "="*60)
    print(f"TRAINING CYCLEGAN: {source_name} → {target_name} (100 EPOCHS)")
    print("="*60)

    history = {'g_loss': [], 'd_loss': []}

    for epoch in range(epochs):
        G_AtoB.train()
        G_BtoA.train()
        D_A.train()
        D_B.train()

        epoch_g_loss = 0
        epoch_d_loss = 0

        loop = tqdm(zip(source_loader, target_loader),
                   desc=f"Epoch [{epoch+1}/{epochs}]",
                   total=min(len(source_loader), len(target_loader)))

        for real_A, real_B in loop:
            real_A = real_A.to(device)
            real_B = real_B.to(device)
            batch_size = real_A.size(0)

            # Create valid and fake labels with correct patch size
            valid = torch.ones(batch_size, 1, patch_size, patch_size).to(device)
            fake = torch.zeros(batch_size, 1, patch_size, patch_size).to(device)

            # ============ Train Generators ============
            optimizer_G.zero_grad()

            # Identity loss
            loss_id_A = criterion_identity(G_BtoA(real_A), real_A) * lambda_identity
            loss_id_B = criterion_identity(G_AtoB(real_B), real_B) * lambda_identity

            # GAN loss
            fake_B = G_AtoB(real_A)
            loss_GAN_AB = criterion_GAN(D_B(fake_B), valid)
            fake_A = G_BtoA(real_B)
            loss_GAN_BA = criterion_GAN(D_A(fake_A), valid)

            # Cycle consistency loss
            recov_A = G_BtoA(fake_B)
            loss_cycle_A = criterion_cycle(recov_A, real_A) * lambda_cycle
            recov_B = G_AtoB(fake_A)
            loss_cycle_B = criterion_cycle(recov_B, real_B) * lambda_cycle

            # Total generator loss
            loss_G = loss_id_A + loss_id_B + loss_GAN_AB + loss_GAN_BA + loss_cycle_A + loss_cycle_B
            loss_G.backward()
            optimizer_G.step()

            # ============ Train Discriminator A ============
            optimizer_D_A.zero_grad()

            # Real loss
            loss_D_A_real = criterion_GAN(D_A(real_A), valid)

            # Fake loss
            fake_A = fake_A_buffer.push_and_pop(fake_A)
            loss_D_A_fake = criterion_GAN(D_A(fake_A.detach()), fake)

            loss_D_A = (loss_D_A_real + loss_D_A_fake) / 2
            loss_D_A.backward()
            optimizer_D_A.step()

            # ============ Train Discriminator B ============
            optimizer_D_B.zero_grad()

            # Real loss
            loss_D_B_real = criterion_GAN(D_B(real_B), valid)

            # Fake loss
            fake_B = fake_B_buffer.push_and_pop(fake_B)
            loss_D_B_fake = criterion_GAN(D_B(fake_B.detach()), fake)

            loss_D_B = (loss_D_B_real + loss_D_B_fake) / 2
            loss_D_B.backward()
            optimizer_D_B.step()

            loss_D = (loss_D_A + loss_D_B).item()

            epoch_g_loss += loss_G.item()
            epoch_d_loss += loss_D

            loop.set_postfix(G_loss=f"{loss_G.item():.2f}", D_loss=f"{loss_D:.2f}")

        history['g_loss'].append(epoch_g_loss / min(len(source_loader), len(target_loader)))
        history['d_loss'].append(epoch_d_loss / min(len(source_loader), len(target_loader)))

        # Save checkpoint every 20 epochs
        if (epoch + 1) % 20 == 0:
            torch.save(G_AtoB.state_dict(), os.path.join(save_dir, f'generator_AtoB_epoch_{epoch+1}.pth'))
            print(f"\n✅ Checkpoint saved at epoch {epoch+1}")

    # Save final model
    torch.save(G_AtoB.state_dict(), os.path.join(save_dir, 'generator_AtoB_final.pth'))
    print(f"\n✅ Final model saved to {save_dir}/generator_AtoB_final.pth")

    return G_AtoB, history


# ============================================
# EVALUATION FUNCTION
# ============================================

def evaluate_cyclegan(generator, source_name, target_name):
    """Evaluate CycleGAN and print metrics"""

    generator.eval()
    results = []

    # Get source class index
    class_names_list = list(cfg.class_names.values())
    source_idx = class_names_list.index(source_name)

    # Load test dataset
    test_dataset = MedicalTestDataset(cfg.dataset_path, cfg.img_size)
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0, drop_last=True)

    print(f"\nEvaluating: {source_name} → {target_name}")

    with torch.no_grad():
        for real_imgs, labels, paths in test_loader:
            real_imgs = real_imgs.to(device)
            mask = labels == source_idx
            if mask.sum() == 0:
                continue

            source_imgs = real_imgs[mask]
            fake_imgs = generator(source_imgs)

            for j in range(min(source_imgs.size(0), 10)):
                real = source_imgs[j].cpu()
                fake = fake_imgs[j].cpu()

                # Denormalize
                real = real * 0.5 + 0.5
                fake = fake * 0.5 + 0.5
                real = torch.clamp(real, 0, 1)
                fake = torch.clamp(fake, 0, 1)

                real_np = real.numpy().transpose(1, 2, 0)
                fake_np = fake.numpy().transpose(1, 2, 0)

                mse = np.mean((real_np - fake_np) ** 2)
                psnr_val = 20 * np.log10(1.0 / np.sqrt(mse)) if mse > 0 else float('inf')
                ssim_val = ssim(real_np, fake_np, channel_axis=2, data_range=1.0)

                results.append({'psnr': psnr_val, 'mse': mse, 'ssim': ssim_val})

            if len(results) >= 100:
                break

    if len(results) == 0:
        print("WARNING: No images found for evaluation!")
        return 0, 0, 0

    avg_psnr = np.mean([r['psnr'] for r in results])
    avg_mse = np.mean([r['mse'] for r in results])
    avg_ssim = np.mean([r['ssim'] for r in results])

    print(f"\n{'='*50}")
    print(f"CYCLEGAN RESULTS (100 EPOCHS)")
    print(f"{'='*50}")
    print(f"Translation: {source_name} → {target_name}")
    print(f"Average PSNR: {avg_psnr:.2f} dB")
    print(f"Average MSE:  {avg_mse:.6f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Total images evaluated: {len(results)}")
    print(f"{'='*50}")

    # Save results
    save_dir = '/content/drive/MyDrive/CSE720/cyclegan_results'
    os.makedirs(save_dir, exist_ok=True)

    with open(os.path.join(save_dir, 'metrics_100epochs.txt'), 'w') as f:
        f.write(f"CycleGAN Results (100 epochs)\n")
        f.write(f"Translation: {source_name} → {target_name}\n")
        f.write(f"{'='*40}\n")
        f.write(f"PSNR: {avg_psnr:.2f} dB\n")
        f.write(f"MSE: {avg_mse:.6f}\n")
        f.write(f"SSIM: {avg_ssim:.4f}\n")

    return avg_psnr, avg_mse, avg_ssim


# ============================================
# RUN CYCLEGAN
# ============================================

if __name__ == "__main__":
    source_domain = "Healthy"
    target_domain = "Diabetic Retinopathy"

    print("\n" + "="*60)
    print(f"CYCLEGAN: {source_domain} → {target_domain}")
    print("="*60)

    # Train
    generator, history = train_cyclegan(source_domain, target_domain, epochs=100)

    # Evaluate
    psnr_val, mse_val, ssim_val = evaluate_cyclegan(generator, source_domain, target_domain)

    print(f"\n{'='*50}")
    print("📊 RECORD THESE VALUES FOR YOUR COMPARISON TABLE:")
    print(f"{'='*50}")
    print(f"   CycleGAN - PSNR: {psnr_val:.2f} dB")
    print(f"   CycleGAN - SSIM: {ssim_val:.4f}")
    print(f"   CycleGAN - MSE: {mse_val:.6f}")
    print(f"{'='*50}")

Using device: cuda
Mounted at /content/drive

CYCLEGAN: Healthy → Diabetic Retinopathy
Loaded 500 images from /content/drive/MyDrive/CSE720/EyeGAN/Healthy
Loaded 500 images from /content/drive/MyDrive/CSE720/EyeGAN/Diabetic Retinopathy
Source (Healthy) training samples: 500
Target (Diabetic Retinopathy) training samples: 500

Generator parameters: 7,837,699
Discriminator parameters: 2,764,737
Discriminator output patch size: 3x3

TRAINING CYCLEGAN: Healthy → Diabetic Retinopathy (100 EPOCHS)


Epoch [20/100]: 100%|██████████| 125/125 [01:06<00:00,  1.89it/s, D_loss=0.20, G_loss=2.89]



✅ Checkpoint saved at epoch 20


Epoch [40/100]: 100%|██████████| 125/125 [01:06<00:00,  1.88it/s, D_loss=0.49, G_loss=2.25]



✅ Checkpoint saved at epoch 40


...


✅ Checkpoint saved at epoch 60


Epoch [80/100]: 100%|██████████| 125/125 [01:06<00:00,  1.87it/s, D_loss=0.57, G_loss=1.99]



✅ Checkpoint saved at epoch 80


Epoch [100/100]: 100%|██████████| 125/125 [01:06<00:00,  1.87it/s, D_loss=0.13, G_loss=2.43]



✅ Checkpoint saved at epoch 100

✅ Final model saved to /content/drive/MyDrive/CSE720/cyclegan_results/generator_AtoB_final.pth
Test dataset: 250 images

Evaluating: Healthy → Diabetic Retinopathy

CYCLEGAN RESULTS (100 EPOCHS)
Translation: Healthy → Diabetic Retinopathy
Average PSNR: 22.00 dB
Average MSE:  0.011977
Average SSIM: 0.7271
Total images evaluated: 50

📊 RECORD THESE VALUES FOR YOUR COMPARISON TABLE:
   CycleGAN - PSNR: 22.00 dB
   CycleGAN - SSIM: 0.7271
   CycleGAN - MSE: 0.011977
